In [9]:
import csv

def format_float(value):
    try:
        return f"{float(value):.2f}"
    except:
        return value  # Si ce n'est pas un float, retourne la valeur d'origine

def format_int(value):
    try:
        return str(int(round(float(value))))
    except:
        return value

def generate_latex_longtable(csv_file_path, output_txt_path):
    with open(csv_file_path, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        rows = list(reader)

    # Colonnes visibles
    headers = ["Name", "RA", "Dec", "Distance", "Ref. Dis", "V_h ± e", "Ref. V_h"]

    latex_lines = []

    latex_lines.append(r"\begin{center}")
    latex_lines.append(r"\setlength{\tabcolsep}{6pt}")
    latex_lines.append(r"\renewcommand{\arraystretch}{1.1}")
    latex_lines.append(r"\begin{longtable}{lrrrllr}")
    latex_lines.append(r"\caption{Liste des galaxies avec coordonnées, distance et vitesse héliocentrique.} \\")
    latex_lines.append(r"\hline")
    latex_lines.append(" & ".join(headers) + r" \\")
    latex_lines.append(r"\hline")
    latex_lines.append(r"\endfirsthead")

    latex_lines.append(r"\hline")
    latex_lines.append(" & ".join(headers) + r" \\")
    latex_lines.append(r"\hline")
    latex_lines.append(r"\endhead")

    latex_lines.append(r"\hline")
    latex_lines.append(r"\endfoot")

    for row in rows:
        name = row["Name"]
        ra = format_float(row["RA"])
        dec = format_float(row["Dec"])

        # Distance fusionnée
        dis = format_float(row["Dis"])
        e_dis_min = format_float(row["e_Dis_min"])
        e_dis_max = format_float(row["e_Dis_max"])
        dis_combined = f"${dis}^{{+{e_dis_max}}}_{{-{e_dis_min}}}$"

        # V_h ± e_V_LG (arrondis à l'entier)
        vh = format_int(row["V_h"])
        e_vlg = format_int(row["e_V_LG"])
        vh_combined = f"${vh} \\pm {e_vlg}$"

        # Références
        ref_dis_val = "CF4" if "CF4" in row["ref_dis"] else "MUSE"
        ref_vh_val = "LEDA" if "LEDA" in row["ref_V_h"] else "MUSE"

        # Ligne du tableau
        line = f"{name} & {ra} & {dec} & {dis_combined} & {ref_dis_val} & {vh_combined} & {ref_vh_val} \\\\"
        latex_lines.append(line)

    latex_lines.append(r"\end{longtable}")
    latex_lines.append(r"\end{center}")

    with open(output_txt_path, 'w', encoding='utf-8') as outfile:
        outfile.write("\n".join(latex_lines))

    print(f"LaTeX longtable (portrait) with formatted values written to {output_txt_path}")


In [10]:
generate_latex_longtable("new_data_lg.csv", "tabl_latex.txt")

LaTeX longtable (portrait) with formatted values written to tabl_latex.txt


In [2]:
import pandas as pd
import argparse

def calculer_statistiques(fichier_csv, colonne):
    try:
        # Lire le fichier CSV
        df = pd.read_csv(fichier_csv)

        # Vérifier que la colonne existe
        if colonne not in df.columns:
            raise ValueError(f"La colonne '{colonne}' n'existe pas dans le fichier.")

        # Extraire les valeurs de la colonne
        valeurs = df[colonne]

        # Calculer les statistiques
        max_val = valeurs.max()
        min_val = valeurs.min()
        median_val = valeurs.median()
        moyenne_val = valeurs.mean()

        return max_val, min_val, median_val, moyenne_val

    except Exception as e:
        print(f"Erreur : {e}")
        return None, None, None, None

In [6]:
calculer_statistiques("new_data_lg_without_muse.csv", "V_h")


(np.float64(1352.9823866604595),
 np.float64(-10.63572754312996),
 np.float64(386.3495026105897),
 np.float64(447.3389928444477))